# Module 00 — Image Processing Fundamentals

Welcome to the first module of **Computer Vision 101**.  We begin without any neural networks — just pixels, kernels, and transforms.  Building this intuition now will make every subsequent module much clearer.

**Topics covered**
1. Images as NumPy arrays
2. Colour space conversions
3. Histograms and CLAHE
4. Convolution from scratch
5. Edge detection (Sobel, Canny)
6. Morphological operations
7. Frequency domain (2-D FFT)
8. Exercise 1 — Unsharp masking
9. Exercise 2 — Simple background subtraction
10. Exercise 3 — CLAHE from first principles

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import urllib.request, io

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# Helper: fetch a small test image from the web
def fetch_image(url, size=(256, 256)):
    with urllib.request.urlopen(url) as resp:
        data = resp.read()
    img = Image.open(io.BytesIO(data)).convert('RGB').resize(size)
    return np.array(img)

# Use a freely available test image
URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
try:
    img_rgb = fetch_image(URL)
except Exception:
    # Fallback: create a synthetic test image
    img_rgb = np.zeros((256, 256, 3), dtype=np.uint8)
    img_rgb[:128, :128] = [220, 60, 60]
    img_rgb[:128, 128:] = [60, 180, 60]
    img_rgb[128:, :128] = [60, 60, 220]
    img_rgb[128:, 128:] = [180, 180, 60]
    from PIL import ImageDraw, ImageFont
    pil = Image.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil)
    draw.text((80, 110), 'TEST', fill=(255,255,255))
    img_rgb = np.array(pil)

print('Image shape:', img_rgb.shape, '| dtype:', img_rgb.dtype)
plt.figure(figsize=(4,4))
plt.imshow(img_rgb)
plt.title('Test image')
plt.axis('off')
plt.show()

## 1. Images as NumPy arrays

A colour image is stored as a 3-D array of shape `(H, W, C)`.  Pixel values are
unsigned 8-bit integers (0–255).  We can slice channels, compute statistics, and
manipulate pixels using standard NumPy operations.

In [ ]:
print('Shape:', img_rgb.shape)        # (H, W, 3)
print('dtype:', img_rgb.dtype)        # uint8
print('Min / Max:', img_rgb.min(), img_rgb.max())
print('Mean per channel (R,G,B):', img_rgb.mean(axis=(0,1)).round(1))

# Split into individual channels
R, G, B = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, ch, name, cmap in zip(axes,
                               [img_rgb, R, G, B],
                               ['RGB', 'Red', 'Green', 'Blue'],
                               [None, 'Reds_r', 'Greens_r', 'Blues_r']):
    ax.imshow(ch, cmap=cmap)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Colour Space Conversions

Different colour spaces highlight different properties of an image.
OpenCV provides fast implementations via `cv2.cvtColor`.

In [ ]:
# Convert RGB → various colour spaces
img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
img_lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

titles = ['Greyscale', 'HSV (Hue)', 'HSV (Saturation)', 'HSV (Value)',
          'LAB (L*)', 'LAB (a*)', 'LAB (b*)']
images = [img_gray,
          img_hsv[:,:,0], img_hsv[:,:,1], img_hsv[:,:,2],
          img_lab[:,:,0], img_lab[:,:,1], img_lab[:,:,2]]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes[0,0].imshow(img_rgb); axes[0,0].set_title('Original RGB'); axes[0,0].axis('off')
for ax, im, t in zip(axes.flat[1:], images, titles):
    ax.imshow(im, cmap='gray')
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Histograms and CLAHE

An image histogram plots the frequency of each pixel intensity value.
*Histogram equalisation* spreads intensities more uniformly, improving contrast.
*CLAHE* (Contrast-Limited Adaptive Histogram Equalisation) applies equalisation
locally, preventing over-amplification of noise.

In [ ]:
# Global histogram equalisation
eq_global = cv2.equalizeHist(img_gray)

# CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
eq_clahe = clahe.apply(img_gray)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, im, title in zip(axes[0],
                         [img_gray, eq_global, eq_clahe],
                         ['Original', 'Global EQ', 'CLAHE']):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title); ax.axis('off')

for ax, im, title in zip(axes[1],
                         [img_gray, eq_global, eq_clahe],
                         ['Histogram — Original', 'Histogram — Global EQ', 'Histogram — CLAHE']):
    ax.hist(im.flatten(), bins=256, range=(0,255), color='steelblue', density=True)
    ax.set_title(title)
    ax.set_xlabel('Intensity'); ax.set_ylabel('Density')

plt.tight_layout()
plt.show()

## 4. Convolution from Scratch

Let us implement a 2-D convolution without using any library function.
This makes explicit what `cv2.filter2D` does under the hood.

In [ ]:
def convolve2d_scratch(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """2-D cross-correlation with reflect padding."""
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image.astype(float), ((ph, ph), (pw, pw)), mode='reflect')
    h, w = image.shape
    out = np.zeros((h, w), dtype=np.float64)
    for i in range(h):
        for j in range(w):
            out[i, j] = (padded[i:i+kh, j:j+kw] * kernel).sum()
    return out

# Gaussian kernel
def gaussian_kernel(size: int = 5, sigma: float = 1.0) -> np.ndarray:
    ax = np.linspace(-(size // 2), size // 2, size)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return k / k.sum()

gk = gaussian_kernel(5, 1.5)
small = img_gray[:64, :64].copy()  # small crop for speed demo
blurred_scratch = convolve2d_scratch(small, gk)
blurred_cv2 = cv2.GaussianBlur(small, (5,5), 1.5)

print('Max difference (scratch vs cv2):', np.abs(blurred_scratch - blurred_cv2).max())

# Apply to full image using cv2 (faster)
blurred_full = cv2.GaussianBlur(img_gray, (15, 15), 3)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_gray, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(blurred_full, cmap='gray'); axes[1].set_title('Gaussian blur (σ=3)'); axes[1].axis('off')
axes[2].imshow(gk, cmap='hot'); axes[2].set_title('Gaussian kernel (5×5)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 5. Edge Detection — Sobel and Canny

Edges correspond to rapid intensity changes.  The **Sobel operator** computes
horizontal and vertical gradients.  **Canny** extends this with non-maximum
suppression and hysteresis thresholding to produce thin, well-localised edges.

In [ ]:
# Sobel gradients
sobelx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobelx**2 + sobely**2)
sobel_dir = np.arctan2(sobely, sobelx) * 180 / np.pi

# Canny edge detector
edges_canny = cv2.Canny(img_gray, threshold1=50, threshold2=150)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, im, t in zip(axes,
    [img_gray, sobelx, sobely, sobel_mag, edges_canny],
    ['Original', 'Sobel X', 'Sobel Y', 'Gradient magnitude', 'Canny edges']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Morphological Operations

Morphological operators work on the *shape* of bright (foreground) regions.
We first threshold the image to get a binary mask, then apply the operators.

In [ ]:
# Otsu's automatic thresholding
_, binary = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

kernel_3x3 = np.ones((3, 3), np.uint8)
kernel_5x5 = np.ones((5, 5), np.uint8)

eroded    = cv2.erode(binary, kernel_3x3, iterations=2)
dilated   = cv2.dilate(binary, kernel_3x3, iterations=2)
opened    = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel_5x5)
closed    = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_5x5)
gradient  = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel_3x3)

titles = ['Binary', 'Eroded', 'Dilated', 'Opened', 'Closed', 'Gradient']
images = [binary, eroded, dilated, opened, closed, gradient]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, im, t in zip(axes.flat, images, titles):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Frequency Domain — 2-D FFT

The 2-D Discrete Fourier Transform decomposes an image into spatial
frequency components.  We can design frequency-domain filters by
multiplying the DFT by a mask and transforming back.

In [ ]:
f = np.fft.fft2(img_gray.astype(float))
fshift = np.fft.fftshift(f)
magnitude = np.log1p(np.abs(fshift))

# Low-pass filter: keep only the central region
H, W = img_gray.shape
cy, cx = H // 2, W // 2
radius = 30
mask_lp = np.zeros((H, W), dtype=bool)
y, x = np.ogrid[:H, :W]
mask_lp[(y - cy)**2 + (x - cx)**2 <= radius**2] = True

fshift_lp = fshift * mask_lp
img_lp = np.abs(np.fft.ifft2(np.fft.ifftshift(fshift_lp))).clip(0, 255).astype(np.uint8)

# High-pass filter: remove the central region
fshift_hp = fshift * ~mask_lp
img_hp = np.abs(np.fft.ifft2(np.fft.ifftshift(fshift_hp))).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, t in zip(axes,
    [img_gray, magnitude, img_lp, img_hp],
    ['Original', 'FFT Magnitude', 'Low-pass (blur)', 'High-pass (edges)']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

## Exercise 1 — Unsharp Masking

**Unsharp masking** sharpens an image by adding a scaled version of the
*high-frequency* content (original minus blurred) back to the original:

```
sharpened = original + amount * (original - blurred)
           = (1 + amount) * original - amount * blurred
```

**Task:** implement `unsharp_mask(image, sigma, amount)` and apply it to
`img_gray`.  Try `sigma=2, amount=1.5`.

**Hint:** use `cv2.GaussianBlur` to obtain the blurred version.

In [ ]:
### EXERCISE 1 — Implement unsharp masking

def unsharp_mask(image: np.ndarray, sigma: float = 1.0, amount: float = 1.0) -> np.ndarray:
    """
    Sharpen *image* using unsharp masking.

    Parameters
    ----------
    image  : H×W uint8 greyscale image.
    sigma  : Standard deviation of the Gaussian blur kernel.
    amount : Sharpening strength.

    Returns
    -------
    np.ndarray  uint8 sharpened image.
    """
    # TODO: implement this function
    raise NotImplementedError("Implement unsharp_mask!")


# Test your implementation:
# sharpened = unsharp_mask(img_gray, sigma=2, amount=1.5)
# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# axes[0].imshow(img_gray, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
# axes[1].imshow(sharpened, cmap='gray'); axes[1].set_title('Sharpened'); axes[1].axis('off')
# plt.tight_layout(); plt.show()

## Exercise 2 — Simple Background Subtraction

Background subtraction isolates moving objects by comparing each frame
against a background model.

**Task:** given a list of synthetic *frames* (created below), compute the
temporal **mean** background, subtract it from each frame, threshold the
result, and visualise the foreground masks.

**Hint:** use `np.abs(frame - background) > threshold` to obtain a mask.

In [ ]:
### EXERCISE 2 — Simple background subtraction

# Synthetic sequence: a moving white square on a noisy background
rng = np.random.default_rng(42)
frames = []
for t in range(10):
    frame = (rng.random((128, 128)) * 60 + 100).clip(0, 255).astype(np.uint8)
    x = 20 + t * 8
    frame[40:70, x:x+30] = 220  # moving rectangle
    frames.append(frame)

def subtract_background(frames, threshold=30):
    """
    Detect foreground pixels using mean-background subtraction.

    Parameters
    ----------
    frames    : list of (H, W) uint8 arrays.
    threshold : absolute difference threshold.

    Returns
    -------
    masks : list of (H, W) bool arrays — True where foreground.
    """
    # TODO: implement this function
    raise NotImplementedError("Implement subtract_background!")


# Test:
# masks = subtract_background(frames, threshold=25)
# fig, axes = plt.subplots(2, 5, figsize=(14, 5))
# for t, (ax_f, ax_m) in enumerate(zip(axes[0], axes[1])):
#     ax_f.imshow(frames[t], cmap='gray', vmin=0, vmax=255)
#     ax_f.set_title(f't={t}'); ax_f.axis('off')
#     ax_m.imshow(masks[t], cmap='gray'); ax_m.axis('off')
# plt.tight_layout(); plt.show()

## Exercise 3 — CLAHE from First Principles

**Task:** implement a simplified version of CLAHE.

Steps:
1. Divide the image into an `grid × grid` array of tiles.
2. For each tile compute its histogram and clip it at `clip_limit * tile_size / 256`.
3. Redistribute the clipped excess uniformly across all bins.
4. Compute the CDF and normalise it to produce a lookup table (LUT).
5. Apply each tile's LUT to that tile's pixels (no interpolation for simplicity).
6. Assemble the tiles back into the output image.

Compare your result to `cv2.createCLAHE`.

In [ ]:
### EXERCISE 3 — CLAHE from first principles

def clahe_scratch(image: np.ndarray, clip_limit: float = 2.0, grid: int = 8) -> np.ndarray:
    """
    Simplified CLAHE without bilinear interpolation between tiles.

    Parameters
    ----------
    image      : (H, W) uint8 greyscale image.
    clip_limit : Clip threshold (relative to uniform histogram height).
    grid       : Number of tiles per axis.

    Returns
    -------
    np.ndarray  uint8 equalised image.
    """
    # TODO: implement this function
    raise NotImplementedError("Implement clahe_scratch!")


# Test:
# result = clahe_scratch(img_gray, clip_limit=2.0, grid=8)
# fig, axes = plt.subplots(1, 3, figsize=(12, 4))
# axes[0].imshow(img_gray, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
# axes[1].imshow(result, cmap='gray'); axes[1].set_title('CLAHE (scratch)'); axes[1].axis('off')
# axes[2].imshow(eq_clahe, cmap='gray'); axes[2].set_title('CLAHE (OpenCV)'); axes[2].axis('off')
# plt.tight_layout(); plt.show()